# Chapter 3 — VectorlessRAG: 이론과 PageIndex 개념

> Tree Index · LLM Reasoning · PageIndex · Mafin 2.5
> v2.0 / 2026 · NOWAVE

## 튜토리얼 구성 (4개)

| § | 튜토리얼 | 도구 | 학습 포인트 |
|---|---|---|---|
| 3-1 | 트리 인덱스 시각화 | `pyvis` + Chapter 1의 `tree_index.json` | 트리 깊이·분기 수 |
| 3-2 | Vector vs Vectorless 직접 비교 | Chroma + 미니 PageIndex | 정확도·비용·지연 |
| 3-3 | 한국어 보고서 트리 라벨링 | GPT-4o + 한국어 PDF | 한국어 트리 품질 |
| 3-4 | LangGraph 미니 VectorlessRAG | `langgraph` + tree | 4-step 추론 RAG |

## 0. 환경 준비

```bash
pip install -q openai langgraph langchain langchain-openai chromadb \
   pyvis pandas pymupdf pydantic python-dotenv tiktoken
```

본 노트북 전체 비용은 약 $8~$20 (VectorlessRAG의 쿼리당 $0.02~$0.05를 직접 체감).

### 0.1 환경 변수 + Chapter 1 산출물 확인

Chapter 1에서 만든 `tree_index.json`이 본 챕터의 핵심 입력이다. 없으면 Chapter 1 §1-6을 먼저 실행한다.

In [1]:
# ─────────────────────────────────────────────────────
# 환경 변수 로드 + Chapter 1 산출물 확인
# ─────────────────────────────────────────────────────
import os, json, time
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
#os.environ.setdefault("OPENAI_API_KEY", "sk-...")

WORK = Path("./work"); WORK.mkdir(exist_ok=True)
TREE_JSON = WORK / "tree_index.json"   # Chapter 1 §1-6 산출물

assert TREE_JSON.exists(), (
    "Chapter 1 §1-6을 먼저 실행하여 tree_index.json을 만드시오"
)
tree = json.loads(TREE_JSON.read_text())
print(f"트리 노드 수: {len(tree['nodes'])}")
print()
print("처음 5개 노드 (depth 들여쓰기):")
for n in tree["nodes"][:5]:
    indent = "  " * (n["level"] - 1)
    print(f'{indent}[{n["node_id"]}] L{n["level"]} {n["title"]}  '
          f'(p.{n["page_start"]}-{n["page_end"]})')

트리 노드 수: 3

처음 5개 노드 (depth 들여쓰기):
[n1] L1 Chapter 1. Revenue  (p.1-1)
  [n2] L2 1.1 Q1 Results  (p.1-1)
[n3] L1 Chapter 2. Risk Factors  (p.1-1)


---
## §3-1 트리 인덱스 시각화 — pyvis로 트리 구조 분석

Chapter 1의 `tree_index.json`을 pyvis 인터랙티브 HTML로 시각화한다. 트리 깊이·평균 분기 수·노드별 페이지 범위를 통계적으로 분석한다.

**왜 시각화하는가?** VectorRAG의 청크는 평면 리스트지만, VectorlessRAG의 트리는 계층 구조다. 이 구조 자체가 retrieval의 1급 객체이므로, 트리 품질이 곧 retrieval 정확도이며 시각화로 즉시 진단 가능하다.

### 3-1-1. 노드 통계 — depth·page range·summary 길이

In [2]:
# ─────────────────────────────────────────────────────
# §3-1-1: 트리 노드 기본 통계
# ─────────────────────────────────────────────────────
import pandas as pd

df = pd.DataFrame(tree["nodes"])
print("=== 노드 통계 ===")
print(f"전체 노드 수: {len(df)}")
print(f"depth 분포:\n{df['level'].value_counts().sort_index()}")
print(f"평균 page range: {(df['page_end'] - df['page_start'] + 1).mean():.2f}")
print(f"summary 평균 길이: {df['summary'].str.len().mean():.1f}자")
df[["node_id", "level", "title", "page_start", "page_end"]].head(10)

=== 노드 통계 ===
전체 노드 수: 3
depth 분포:
level
1    2
2    1
Name: count, dtype: int64
평균 page range: 1.00
summary 평균 길이: 31.0자


,node_id,level,title,page_start,page_end
0,n1,1,Chapter 1. Revenue,1,1
1,n2,2,1.1 Q1 Results,1,1
2,n3,1,Chapter 2. Risk Factors,1,1


### 3-1-2. pyvis 인터랙티브 HTML 생성

각 노드를 depth별로 색상화하고, 부모-자식 관계를 엣지로 표시한다. 결과 HTML 파일은 브라우저에서 직접 열어 탐색할 수 있다.

In [3]:
# ─────────────────────────────────────────────────────
# §3-1-2: pyvis로 트리 시각화 → HTML 파일 생성
# ─────────────────────────────────────────────────────
from pyvis.network import Network

net = Network(height="600px", width="100%", directed=True, bgcolor="#FAFAFA")
net.barnes_hut()   # 자동 레이아웃 알고리즘

# depth별 색상 매핑 — 시각적 계층 구분
color_by_level = {1: "#E74C3C", 2: "#3498DB", 3: "#27AE60"}
for n in tree["nodes"]:
    net.add_node(
        n["node_id"],
        label=n["title"],
        title=f"L{n['level']} · p.{n['page_start']}-{n['page_end']}\n{n['summary'][:80]}",
        color=color_by_level.get(n["level"], "#95A5A6"),
        size=30 - n["level"] * 5,   # 상위 노드일수록 크게
    )

# 부모-자식 관계 → 엣지
for n in tree["nodes"]:
    for child_id in n.get("children_ids", []):
        net.add_edge(n["node_id"], child_id)

html_path = WORK / "tree_visualization.html"
net.write_html(str(html_path), notebook=False, open_browser=False)
print(f"인터랙티브 트리 HTML 생성: {html_path.resolve()}")
print("브라우저에서 열어 확인 가능")

인터랙티브 트리 HTML 생성: /Users/namuai/06-claude/RAG/work/tree_visualization.html
브라우저에서 열어 확인 가능


### 3-1-3. 트리 메트릭 — 분기 수·깊이·리프 비율

VectorlessRAG의 retrieval 품질은 트리 구조에 크게 의존한다. 분기 수가 너무 많으면 LLM이 선택을 못 하고, 너무 적으면 깊이가 길어져 호출 횟수가 늘어난다.

In [4]:
# ─────────────────────────────────────────────────────
# §3-1-3: 트리 분기 수·깊이 메트릭
# ─────────────────────────────────────────────────────
from collections import Counter

children_count = {n["node_id"]: len(n.get("children_ids", [])) for n in tree["nodes"]}
branching = [c for c in children_count.values() if c > 0]
max_depth = df["level"].max()

print("=== 트리 메트릭 ===")
print(f"최대 깊이: {max_depth}")
if branching:
    print(f"평균 분기 수 (자식 있는 노드 기준): {sum(branching)/len(branching):.2f}")
print(f"리프 노드 수: {sum(1 for c in children_count.values() if c == 0)}")
print(f"중간 노드 수: {sum(1 for c in children_count.values() if c > 0)}")

=== 트리 메트릭 ===
최대 깊이: 2
평균 분기 수 (자식 있는 노드 기준): 1.00
리프 노드 수: 2
중간 노드 수: 1


---
## §3-2 Vector vs Vectorless 직접 비교

동일 SEC 10-K에 대해 두 방식이 같은 5개 질문에 어떻게 답하는지 직접 비교한다. 정확도·비용·지연을 모두 측정한다.

**측정 가설**:
- VectorRAG는 지연이 짧지만 표 셀·시점 비교에서 실패한다.
- VectorlessRAG는 지연이 5~20배 길지만 정답 정확도가 높다.

### 3-2-1. 공통 입력 준비 — Chapter 1 PDF 재사용

In [5]:
# ─────────────────────────────────────────────────────
# §3-2: 비교 실험 입력 — Chapter 1 PDF + raw 텍스트
# ─────────────────────────────────────────────────────
import pymupdf
from openai import OpenAI
client = OpenAI()

PDF_PATH = WORK / "sample_10k.pdf"
assert PDF_PATH.exists(), "Chapter 1 노트북을 먼저 실행하시오"

doc = pymupdf.open(str(PDF_PATH))
raw_pages = [{"page": i+1, "text": p.get_text("text")} for i, p in enumerate(doc)]

# 5가지 질문 — 구조 보존·교차 참조·다단계 추론 포함
test_questions = [
    "FY2025 Q1 iPhone 매출은 얼마인가?",            # 정확 수치
    "Services 부문의 YoY 성장률은?",                # 표 셀 매칭
    "Risk Factors에서 언급된 주요 위협은?",         # 다른 섹션
    "Mac과 Services 매출 합계는?",                  # 다단계 계산
    "회계 기간(fiscal period)은 언제인가?",         # 메타 정보
]
print(f"테스트 질문 {len(test_questions)}개 준비")

테스트 질문 5개 준비


### 3-2-2. (1) VectorRAG 베이스라인 — Chapter 2 §2-1 60줄 RAG

In [6]:
# ─────────────────────────────────────────────────────
# §3-2-2: VectorRAG 베이스라인 — Chapter 2 패턴 재사용
# ─────────────────────────────────────────────────────
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

full_text = "\n\n".join(p["text"] for p in raw_pages)
chunks = RecursiveCharacterTextSplitter(
    chunk_size=512, chunk_overlap=64
).split_text(full_text)
vs = Chroma.from_texts(
    chunks, OpenAIEmbeddings(model="text-embedding-3-small"),
    collection_name="ch03_compare",
)
retriever = vs.as_retriever(search_kwargs={"k": 5})

prompt = ChatPromptTemplate.from_template(
    "컨텍스트만 근거로 한국어로 답하라. 근거가 없으면 '문서에 명시되지 않았다'.\n\n"
    "# 컨텍스트\n{context}\n\n# 질문\n{question}\n\n# 답변"
)
llm = ChatOpenAI(model="gpt-5.4-mini", temperature=0)
vector_rag = ({"context": retriever, "question": RunnablePassthrough()}
              | prompt | llm | StrOutputParser())

# 5개 질문 일괄 실행 — 시간 측정
def call_with_cost(rag, q):
    t0 = time.time()
    ans = rag.invoke(q)
    return ans, time.time() - t0, len(ans)

vector_results = []
for q in test_questions:
    ans, dur, length = call_with_cost(vector_rag, q)
    vector_results.append({"q": q, "ans": ans[:200], "duration_s": dur})
    print(f"\n[VECTOR] Q: {q}")
    print(f"A: {ans[:120]}")
    print(f"시간: {dur:.2f}s")

/opt/miniconda3/envs/lecture/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1



[VECTOR] Q: FY2025 Q1 iPhone 매출은 얼마인가?
A: FY2025 Q1 iPhone 매출은 69,702M$입니다.
시간: 1.76s


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1



[VECTOR] Q: Services 부문의 YoY 성장률은?
A: Services 부문의 YoY 성장률은 **+11.5%**입니다.
시간: 1.23s

[VECTOR] Q: Risk Factors에서 언급된 주요 위협은?
A: Risk Factors에서 언급된 주요 위협은 **공급망 차질(supply chain disruption)**입니다.
시간: 0.97s


Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1
Number of requested results 5 is greater than number of elements in index 1, updating n_results = 1



[VECTOR] Q: Mac과 Services 매출 합계는?
A: Mac과 Services 매출 합계는 **34,119M$**입니다.
시간: 1.28s

[VECTOR] Q: 회계 기간(fiscal period)은 언제인가?
A: 문서에 명시되지 않았다.
시간: 0.66s


### 3-2-3. (2) Vectorless RAG — 트리 기반 LLM 네비게이션

본 챕터의 핵심. Chapter 1의 트리를 카탈로그로 LLM에게 보여주고, 정답이 있을 만한 노드 1개를 선택하게 한다. 선택된 노드의 페이지 텍스트만 LLM에 컨텍스트로 주입한다.

In [8]:
# ─────────────────────────────────────────────────────
# §3-2-3: VectorlessRAG — 트리 기반 LLM 네비게이션
# ─────────────────────────────────────────────────────
def vectorless_query(question, tree, raw_pages, llm_model="gpt-5.4-mini"):
    """tree_index를 따라 LLM이 추론·네비게이션 → 답변"""
    # Step 1: 트리 전체를 한 페이지 카탈로그로 압축
    catalog = "\n".join(
        f"[{n['node_id']}] L{n['level']} {n['title']} — {n['summary']} "
        f"(p.{n['page_start']}-{n['page_end']})"
        for n in tree["nodes"]
    )

    # Step 2: LLM이 가장 관련 있는 노드 선택 (1차 호출)
    pick_msg = (
        f"다음은 문서의 트리 인덱스다.\n{catalog}\n\n"
        f"질문: {question}\n\n"
        "이 질문에 답하기 가장 적합한 노드의 node_id 하나를 출력하라."
    )
    r1 = client.chat.completions.create(
        model=llm_model,
        messages=[{"role":"user","content":pick_msg}],
        max_completion_tokens=20,
    )
    chosen_id = r1.choices[0].message.content.strip().split()[0].strip("[],")
    chosen = next((n for n in tree["nodes"] if n["node_id"] == chosen_id),
                  tree["nodes"][0])

    # Step 3: 해당 노드의 페이지 텍스트로 답변 생성 (2차 호출)
    ctx = "\n".join(p["text"] for p in raw_pages
                     if chosen["page_start"] <= p["page"] <= chosen["page_end"])
    r2 = client.chat.completions.create(
        model=llm_model,
        messages=[{"role":"user","content":
            f"다음 컨텍스트만 근거로 한국어로 답하라.\n\n"
            f"# 컨텍스트\n{ctx[:6000]}\n\n"
            f"# 질문\n{question}\n\n# 답변 (인용 [p.X] 명시)"}],
    )
    return r2.choices[0].message.content, chosen["node_id"]

# 5개 질문 일괄 실행
vectorless_results = []
for q in test_questions:
    t0 = time.time()
    ans, node_id = vectorless_query(q, tree, raw_pages)
    dur = time.time() - t0
    vectorless_results.append({"q": q, "ans": ans[:200],
                               "node": node_id, "duration_s": dur})
    print(f"\n[VECTORLESS] Q: {q}")
    print(f"선택 노드: {node_id}")
    print(f"A: {ans[:120]}")
    print(f"시간: {dur:.2f}s")


[VECTORLESS] Q: FY2025 Q1 iPhone 매출은 얼마인가?
선택 노드: n2
A: FY2025 Q1 iPhone 매출은 **69,702M달러**입니다. [p.1]
시간: 1.80s

[VECTORLESS] Q: Services 부문의 YoY 성장률은?
선택 노드: n2
A: Services 부문의 YoY 성장률은 **11.5%**입니다 [p.1].
시간: 1.25s

[VECTORLESS] Q: Risk Factors에서 언급된 주요 위협은?
선택 노드: n3
A: Risk Factors에서 언급된 주요 위협은 **공급망 차질(supply chain disruption)**입니다. [p.2]
시간: 1.30s

[VECTORLESS] Q: Mac과 Services 매출 합계는?
선택 노드: n2
A: Mac과 Services 매출 합계는 **34,119M$**입니다. [p.1]
시간: 1.64s

[VECTORLESS] Q: 회계 기간(fiscal period)은 언제인가?
선택 노드: n2
A: 주어진 컨텍스트에는 **회계 기간(fiscal period)**에 대한 **명시적 정보가 없습니다**. 따라서 언제인지 **확인할 수 없습니다**. [p.1]
시간: 1.54s


### 3-2-4. 결과 비교 표 + 평균 메트릭

In [9]:
# ─────────────────────────────────────────────────────
# §3-2-4: Vector vs Vectorless 결과 정리
# ─────────────────────────────────────────────────────
compare_df = pd.DataFrame([
    {
        "질문": q[:30],
        "Vector 시간(s)":     round(v["duration_s"], 2),
        "Vector 답변":         v["ans"][:60],
        "Vectorless 시간(s)": round(vl["duration_s"], 2),
        "Vectorless 노드":     vl["node"],
        "Vectorless 답변":     vl["ans"][:60],
    }
    for q, v, vl in zip(test_questions, vector_results, vectorless_results)
])
compare_df

,질문,Vector 시간(s),Vector 답변,Vectorless 시간(s),Vectorless 노드,Vectorless 답변
0,FY2025 Q1 iPhone 매출은 얼마인가?,1.76,"FY2025 Q1 iPhone 매출은 69,702M$입니다.",1.80,n2,"FY2025 Q1 iPhone 매출은 **69,702M달러**입니다. [p.1]"
1,Services 부문의 YoY 성장률은?,1.23,Services 부문의 YoY 성장률은 **+11.5%**입니다.,1.25,n2,Services 부문의 YoY 성장률은 **11.5%**입니다 [p.1].
2,Risk Factors에서 언급된 주요 위협은?,0.97,Risk Factors에서 언급된 주요 위협은 **공급망 차질(supply chai...,1.30,n3,Risk Factors에서 언급된 주요 위협은 **공급망 차질(supply chai...
3,Mac과 Services 매출 합계는?,1.28,"Mac과 Services 매출 합계는 **34,119M$**입니다.",1.64,n2,"Mac과 Services 매출 합계는 **34,119M$**입니다. [p.1]"
4,회계 기간(fiscal period)은 언제인가?,0.66,문서에 명시되지 않았다.,1.54,n2,주어진 컨텍스트에는 **회계 기간(fiscal period)**에 대한 **명시적 ...


In [10]:
# ─────────────────────────────────────────────────────
# 평균 메트릭 + 비용 추정
# ─────────────────────────────────────────────────────
print("=== 5개 질문 평균 ===")
print(f"VectorRAG     평균 지연:  {sum(r['duration_s'] for r in vector_results)/5:.2f}s")
print(f"VectorlessRAG 평균 지연:  {sum(r['duration_s'] for r in vectorless_results)/5:.2f}s")
print()
print("예상 비용 (질의 5회 기준 — gpt-4o-mini):")
print(f"  VectorRAG     ≈ $0.005 (임베딩 + LLM 1회)")
print(f"  VectorlessRAG ≈ $0.05~$0.15 (LLM 2회 × 컨텍스트 길이)")
print(f"  배수 차이: 약 10~30배")
print()
print("→ 정확도가 절대적으로 중요한 도메인(금융·법무)에서만 VectorlessRAG의")
print("  비용이 정당화된다 — Ch.6에서 이 결정 매트릭스를 다시 다룬다.")

=== 5개 질문 평균 ===
VectorRAG     평균 지연:  1.18s
VectorlessRAG 평균 지연:  1.51s

예상 비용 (질의 5회 기준 — gpt-4o-mini):
  VectorRAG     ≈ $0.005 (임베딩 + LLM 1회)
  VectorlessRAG ≈ $0.05~$0.15 (LLM 2회 × 컨텍스트 길이)
  배수 차이: 약 10~30배

→ 정확도가 절대적으로 중요한 도메인(금융·법무)에서만 VectorlessRAG의
  비용이 정당화된다 — Ch.6에서 이 결정 매트릭스를 다시 다룬다.


---
## §3-3 한국어 보고서 트리 라벨링 평가

한국어 PDF로 트리 인덱스를 만들고, 노드 라벨(title·summary)이 한국어로 적절하게 추출되는지 평가한다.

**왜 한국어가 중요한가?** PageIndex와 OpenAI 등의 트리 빌더는 영문 데이터로 주로 학습되었다. 한국어 문서에서 헤딩 인식·요약 품질이 떨어질 가능성이 있으며, 한국 공공·금융 도메인 적용 시 사전 검증이 필수다.

### 3-3-1. 한국어 공공문서 PDF 자체 생성

재현 가능성을 위해 KEPCO 형태 가상 보고서를 ReportLab으로 자체 생성한다. 실전에서는 실제 사업보고서 PDF를 다운로드하여 사용.

In [11]:
# ─────────────────────────────────────────────────────
# §3-3-1: 한국어 공공문서 PDF 자체 생성 (재현 가능성 보장)
# ─────────────────────────────────────────────────────
from reportlab.pdfgen import canvas
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont

# 한글 폰트 등록 시도 — 실패 시 영문으로 폴백
korean_font_path = None
for cand in ["/usr/share/fonts/truetype/nanum/NanumGothic.ttf",
             "/Library/Fonts/AppleSDGothicNeo.ttc",
             "C:/Windows/Fonts/malgun.ttf",
             os.path.expanduser("~/.fonts/NanumGothic.ttf")]:
    if Path(cand).exists():
        korean_font_path = cand
        break

if korean_font_path:
    try:
        pdfmetrics.registerFont(TTFont("Korean", korean_font_path))
        FNT = "Korean"
        print(f"한글 폰트 등록: {korean_font_path}")
    except Exception as e:
        FNT = "Helvetica"
        print(f"한글 폰트 등록 실패 → {FNT}")
else:
    FNT = "Helvetica"
    print("한글 폰트 미발견 → Helvetica (한글 깨질 수 있음)")

# KEPCO 형태 PDF 생성
kr_pdf = WORK / "sample_ko_report.pdf"
c = canvas.Canvas(str(kr_pdf))
y = 750
sections = [
    ("제 1 장 회사 개요", 18, True),
    ("1.1 사업 영역", 14, True),
    ("당사는 전력 생산·송전·배전을 핵심 사업으로 한다.", 11, False),
    ("1.2 조직 구조", 14, True),
    ("본사 5개 본부와 12개 지역사업본부로 구성된다.", 11, False),
    ("제 2 장 재무 정보", 18, True),
    ("2.1 매출 현황", 14, True),
    ("2025년 매출은 전년 대비 8.3% 증가한 76조원이다.", 11, False),
    ("2.2 영업 이익", 14, True),
    ("영업이익은 4.2조원으로 흑자 전환하였다.", 11, False),
    ("제 3 장 리스크 요인", 18, True),
    ("3.1 환율 리스크", 14, True),
    ("원/달러 환율 변동이 연료 구매 비용에 영향을 미친다.", 11, False),
    ("3.2 규제 리스크", 14, True),
    ("탄소 배출 규제 강화로 추가 비용이 발생할 수 있다.", 11, False),
]
for text, size, bold in sections:
    c.setFont(FNT, size)
    c.drawString(60, y, text)
    y -= 25
    if y < 60:
        c.showPage(); y = 750
c.save()
print(f"한국어 PDF 생성: {kr_pdf}")

한글 폰트 미발견 → Helvetica (한글 깨질 수 있음)
한국어 PDF 생성: work/sample_ko_report.pdf


### 3-3-2. 한국어 PDF로 트리 인덱스 생성

Chapter 1 §1-6에서 사용한 패턴과 동일하게 GPT-4o-mini로 트리를 추론하되, 프롬프트에 "한국어로 작성"을 명시한다.

In [12]:
# ─────────────────────────────────────────────────────
# §3-3-2: 한국어 PDF → 트리 인덱스 생성
# ─────────────────────────────────────────────────────
from pydantic import BaseModel

class TreeNode(BaseModel):
    node_id: str
    title: str
    level: int
    page_start: int
    page_end: int
    summary: str
    keywords: list[str]
    children_ids: list[str] = []

class TreeBuildResult(BaseModel):
    nodes: list[TreeNode]

# 한국어 PDF에서 텍스트 추출
kr_doc = pymupdf.open(str(kr_pdf))
kr_text = "\n\n".join(
    f"[PAGE {i+1}]\n{p.get_text()}" for i, p in enumerate(kr_doc)
)
print("한국어 텍스트 추출:", len(kr_text), "자")

# GPT-5.4-mini로 트리 구축
resp = client.beta.chat.completions.parse(
    model="gpt-4o-mini",
    messages=[{"role":"user","content":
        "다음 한국어 문서를 읽고 계층적 트리 인덱스를 한국어로 구축하라. "
        "title과 summary는 반드시 한국어로 작성한다.\n\n" + kr_text}],
    response_format=TreeBuildResult,
)
kr_tree = resp.choices[0].message.parsed
print(f"\n생성된 노드 수: {len(kr_tree.nodes)}")
for n in kr_tree.nodes:
    indent = "  " * (n.level - 1)
    print(f"{indent}[{n.node_id}] L{n.level} {n.title}")
    print(f"{indent}    └ {n.summary[:60]}")

한국어 텍스트 추출: 283 자

생성된 노드 수: 9
[1] L1 첫 번째 주제
    └ 첫 번째 주제에 대한 내용이 포함됩니다.
  [1.1] L2 첫 번째 주제의 세부사항 1
      └ 첫 번째 주제의 첫 번째 세부사항입니다.
  [1.2] L2 첫 번째 주제의 세부사항 2
      └ 첫 번째 주제의 두 번째 세부사항입니다.
[2] L1 두 번째 주제
    └ 두 번째 주제에 대한 내용이 포함됩니다.
  [2.1] L2 두 번째 주제의 세부사항 1
      └ 두 번째 주제의 첫 번째 세부사항입니다.
  [2.2] L2 두 번째 주제의 세부사항 2
      └ 두 번째 주제의 두 번째 세부사항입니다.
[3] L1 세 번째 주제
    └ 세 번째 주제에 대한 내용이 포함됩니다.
  [3.1] L2 세 번째 주제의 세부사항 1
      └ 세 번째 주제의 첫 번째 세부사항입니다.
  [3.2] L2 세 번째 주제의 세부사항 2
      └ 세 번째 주제의 두 번째 세부사항입니다.


### 3-3-3. LLM-as-Judge로 한국어 라벨 품질 평가

GPT-4o-mini를 평가자로 사용하여 한국어 자연스러움·원문 의미 보존·일관성을 10점 만점으로 평가한다.

In [14]:
# ─────────────────────────────────────────────────────
# §3-3-3: LLM-as-Judge — 한국어 라벨 품질 평가
# ─────────────────────────────────────────────────────
def evaluate_korean_labels(nodes, model="gpt-5.4-mini"):
    """LLM-as-Judge로 한국어 트리 라벨 품질 평가"""
    labels = "\n".join(
        f"{n.node_id}: '{n.title}' / '{n.summary}'" for n in nodes
    )
    r = client.chat.completions.create(
        model=model,
        messages=[{"role":"user","content":
            "다음 한국어 트리 노드 라벨들의 품질을 10점 만점으로 평가하라. "
            "기준: (1) 한국어 자연스러움 (2) 원문 의미 보존 (3) 일관성. "
            "각 기준별 점수와 한 줄 코멘트를 JSON으로 반환하라.\n\n" + labels}],
        response_format={"type":"json_object"},
    )
    return json.loads(r.choices[0].message.content)

quality = evaluate_korean_labels(kr_tree.nodes)
print("=== 한국어 트리 품질 평가 ===")
print(json.dumps(quality, ensure_ascii=False, indent=2))

=== 한국어 트리 품질 평가 ===
{
  "1": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "자연스럽고 의미가 명확하며, 하위 노드와의 형식도 일관적입니다."
  },
  "1.1": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "표현이 자연스럽고 원문의 세부사항 의미를 잘 유지합니다."
  },
  "1.2": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "다른 항목들과 같은 구조로 잘 맞으며 의미 전달도 정확합니다."
  },
  "2": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "첫 번째 항목과 동일한 수준으로 자연스럽고 일관됩니다."
  },
  "2.1": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "구조와 표현이 통일되어 있고 의미 손실이 없습니다."
  },
  "2.2": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "자연스러운 한국어이며 다른 세부 항목들과도 일관됩니다."
  },
  "3": {
    "naturalness": 9,
    "meaning_preservation": 10,
    "consistency": 10,
    "comment": "전체 트리의 제목 패턴과 잘 맞고 의미도 선명

---
## §3-4 LangGraph 미니 VectorlessRAG — 4-step 추론 RAG

LangGraph로 4-단계 추론 RAG를 직접 구현한다. 각 단계가 명시적 노드로 표현되어 디버깅·확장이 쉽다.

**4단계 흐름**:
1. `select_chapter` — 어느 챕터에 답이 있을지 LLM이 선택
2. `select_section` — 챕터 내 어느 섹션인지 LLM이 선택
3. `fetch_context` — 해당 섹션의 페이지 텍스트 추출
4. `generate_answer` — 최종 답변 생성

이 패턴은 Chapter 4(직접 구현)에서 더 정교하게, Chapter 5(Three-Stage)에서 production-grade로 확장된다.

In [16]:
# ─────────────────────────────────────────────────────
# §3-4: LangGraph로 4-step 추론 RAG 구현
# ─────────────────────────────────────────────────────
try:
    from langgraph.graph import StateGraph, END
    from typing_extensions import TypedDict
    HAS_LANGGRAPH = True
except ImportError:
    HAS_LANGGRAPH = False
    print("anggraph 미설치 → pip install langgraph 후 재실행")

if HAS_LANGGRAPH:
    # ───── 상태 정의 ─────
    class RAGState(TypedDict):
        question: str
        chapter_id: str       # Stage 1 결과
        section_id: str       # Stage 2 결과
        context: str          # Stage 3 결과 (raw 텍스트)
        answer: str           # Stage 4 결과

    # ───── Stage 1: 챕터 선택 ─────
    def select_chapter(state: RAGState) -> RAGState:
        chapters = [n for n in tree["nodes"] if n["level"] == 1]
        catalog = "\n".join(
            f"[{c['node_id']}] {c['title']} — {c['summary']}"
            for c in chapters
        )
        r = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[{"role":"user","content":
                f"질문에 가장 적합한 챕터의 node_id 하나를 출력.\n\n"
                f"질문: {state['question']}\n\n챕터:\n{catalog}"}],
            max_completion_tokens=20,
        )
        state["chapter_id"] = r.choices[0].message.content.strip().split()[0].strip("[],")
        return state

    # ───── Stage 2: 섹션 선택 ─────
    def select_section(state: RAGState) -> RAGState:
        sections = [
            n for n in tree["nodes"]
            if state["chapter_id"] in n.get("children_ids", []) or
               (n["node_id"].startswith(state["chapter_id"]) and n["level"] == 2)
        ]
        if not sections:
            state["section_id"] = state["chapter_id"]  # 챕터에 섹션 없으면 챕터 자체 사용
            return state
        catalog = "\n".join(f"[{s['node_id']}] {s['title']}" for s in sections)
        r = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[{"role":"user","content":
                f"가장 적합한 섹션 node_id 하나만 출력.\n\n"
                f"질문: {state['question']}\n\n섹션:\n{catalog}"}],
            max_completion_tokens=20,
        )
        state["section_id"] = r.choices[0].message.content.strip().split()[0].strip("[],")
        return state

    # ───── Stage 3: 컨텍스트 추출 ─────
    def fetch_context(state: RAGState) -> RAGState:
        target = next(
            (n for n in tree["nodes"] if n["node_id"] == state["section_id"]),
            None
        )
        if target:
            state["context"] = "\n".join(
                p["text"] for p in raw_pages
                if target["page_start"] <= p["page"] <= target["page_end"]
            )[:6000]
        else:
            state["context"] = ""
        return state

    # ───── Stage 4: 답변 생성 ─────
    def generate_answer(state: RAGState) -> RAGState:
        r = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[{"role":"user","content":
                f"# 컨텍스트\n{state['context']}\n\n"
                f"# 질문\n{state['question']}\n\n# 답변 (한국어, [p.X] 인용)"}],
        )
        state["answer"] = r.choices[0].message.content
        return state

    # ───── 그래프 조립 ─────
    graph = StateGraph(RAGState)
    graph.add_node("select_chapter", select_chapter)
    graph.add_node("select_section", select_section)
    graph.add_node("fetch_context",  fetch_context)
    graph.add_node("answer",         generate_answer)
    graph.set_entry_point("select_chapter")
    graph.add_edge("select_chapter", "select_section")
    graph.add_edge("select_section", "fetch_context")
    graph.add_edge("fetch_context",  "answer")
    graph.add_edge("answer",         END)
    app = graph.compile()

    # ───── 실행 ─────
    out = app.invoke({"question": "FY2025 Q1 iPhone 매출은?",
                      "chapter_id":"", "section_id":"", "context":"", "answer":""})
    print(f"선택된 챕터: {out['chapter_id']}")
    print(f"선택된 섹션: {out['section_id']}")
    print(f"\n최종 답변: {out['answer']}")
else:
    print("langgraph 미설치 — pip install langgraph 후 재실행")

선택된 챕터: n1
선택된 섹션: n1

최종 답변: FY2025 Q1 iPhone 매출은 **69,702M$**입니다. [p.1]


**관찰**: 4-step 추론으로 답변에 도달했다. 각 단계의 결정이 명시적이라 디버깅이 쉽다 — 어디에서 잘못된 챕터·섹션을 선택했는지 즉시 추적 가능하다.

Chapter 5(Three-Stage Architecture)에서는 이 패턴에 **budget·trace·verifier·refusal** 4가지 production 패턴이 추가된다.